##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
# Import Required Libraries
import os
import cv2
import random
import numpy as np
import datetime as dt
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline

from collections import deque
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model

In [2]:
# Set seeds
seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

In [3]:
DATASET_DIR = "/Users/sarahalashgar/Downloads/UCF11_updated_mpg"
# Check available classes
print(os.listdir(DATASET_DIR))

['volleyball_spiking', 'golf_swing', 'swing', 'tennis_swing', 'soccer_juggling', 'horse_riding', 'walking', 'basketball', 'trampoline_jumping', 'biking', 'diving']


In [4]:
IMAGE_HEIGHT, IMAGE_WIDTH = 64, 64
MAX_IMAGES_PER_CLASS = 3000
CLASSES_LIST = ["basketball", "biking", "tennis_swing"]

In [5]:
# Preprocess the dataset

def extract_frames(video_path):
    frames_list = []

    video_reader = cv2.VideoCapture(video_path)

    while True:
        success, frame = video_reader.read()

        if not success:
            break

        resized_frame = cv2.resize(frame, (IMAGE_WIDTH, IMAGE_HEIGHT))
        normalized_frame = resized_frame / 255.0

        frames_list.append(normalized_frame)

    video_reader.release()

    return frames_list

In [6]:
def create_dataset():
    features = []
    labels = []

    for class_index, class_name in enumerate(CLASSES_LIST):
        print(f"Processing class: {class_name}")

        class_folder_path = os.path.join(DATASET_DIR, class_name)

        class_frames = []

        for root, dirs, files in os.walk(class_folder_path):
            for file_name in files:

                if not file_name.endswith((".mpg", ".avi", ".mp4")):
                    continue

                video_path = os.path.join(root, file_name)

                frames = extract_frames(video_path)
                class_frames.extend(frames)

        print(f"Frames found for {class_name}: {len(class_frames)}")

        selected_frames_count = min(MAX_IMAGES_PER_CLASS, len(class_frames))

        selected_frames = random.sample(class_frames, selected_frames_count)

        features.extend(selected_frames)
        labels.extend([class_index] * selected_frames_count)

    features = np.array(features)
    labels = np.array(labels)

    return features, labels

In [7]:
features, labels = create_dataset()

one_hot_encoded_labels = to_categorical(labels)

print("Features shape:", features.shape)
print("Labels shape:", one_hot_encoded_labels.shape)

Processing class: basketball
Frames found for basketball: 19230
Processing class: biking
Frames found for biking: 32863
Processing class: tennis_swing
Frames found for tennis_swing: 26574
Features shape: (9000, 64, 64, 3)
Labels shape: (9000, 3)


In [8]:
# Split the data into training and testing

features_train, features_test, labels_train, labels_test = train_test_split(
    features,
    one_hot_encoded_labels,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("Training data:", features_train.shape)
print("Testing data:", features_test.shape)

Training data: (7200, 64, 64, 3)
Testing data: (1800, 64, 64, 3)


In [9]:
# Create the model

model = Sequential()

model.add(Conv2D(64, (3, 3), activation="relu", input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3)))
model.add(Conv2D(64, (3, 3), activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(GlobalAveragePooling2D())

model.add(Dense(256, activation="relu"))
model.add(BatchNormalization())

model.add(Dense(len(CLASSES_LIST), activation="softmax"))

model.summary()

/opt/anaconda3/envs/gensim-env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 60, 60, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,411 (224.26 KB)

 Trainable params: 56,771 (221.76 KB)

 Non-trainable params: 640 (2.50 KB)

In [11]:
# Train the model

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

history = model.fit(
    features_train,
    labels_train,
    epochs=10,
    batch_size=4,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stopping])

Epoch 1/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 26s 17ms/step - accuracy: 0.5729 - loss: 0.9111 - val_accuracy: 0.5785 - val_loss: 1.0891
Epoch 2/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7342 - loss: 0.6365 - val_accuracy: 0.6153 - val_loss: 0.9963
Epoch 3/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 25s 17ms/step - accuracy: 0.8007 - loss: 0.5037 - val_accuracy: 0.6972 - val_loss: 0.8577
Epoch 4/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.8347 - loss: 0.4335 - val_accuracy: 0.9035 - val_loss: 0.2630
Epoch 5/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.8580 - loss: 0.3709 - val_accuracy: 0.9056 - val_loss: 0.2446
Epoch 6/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.8764 - loss: 0.3309 - val_accuracy: 0.9632 - val_loss: 0.1188
Epoch 7/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.8965 - loss: 0.2942 - val_accuracy: 0.8340 - val_loss: 0.4101
Epoch 8/10
1440/1440 ━━━━━━━━━━━━━━━━━━━━ 27s 19ms/step - accuracy: 0.9016 -

In [12]:
# Test the model

loss, accuracy = model.evaluate(features_test, labels_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9572 - loss: 0.1348
Test Loss: 0.13483786582946777
Test Accuracy: 0.9572222232818604


In [13]:
# Save the trained model with your name

student_name = "SarahAlashgar"

save_path = f"{student_name}_ucf11_model.h5"

model.save(save_path)

print(f"Model saved as {save_path}")

Model saved as SarahAlashgar_ucf11_model.h5


In [14]:
def predict_youtube_video(video_path, frames_count=50):
    video_reader = cv2.VideoCapture(video_path)

    if not video_reader.isOpened():
        print("Could not open video:", video_path)
        return

    total_frames = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        print("No frames found in video:", video_path)
        video_reader.release()
        return

    skip_frames = max(total_frames // frames_count, 1)

    predictions = []

    for frame_counter in range(frames_count):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames)

        success, frame = video_reader.read()

        if not success:
            continue

        resized_frame = cv2.resize(frame, (IMAGE_WIDTH, IMAGE_HEIGHT))
        normalized_frame = resized_frame / 255.0

        prediction = model.predict(
            np.expand_dims(normalized_frame, axis=0),
            verbose=0
        )

        predictions.append(prediction[0])

    video_reader.release()

    if len(predictions) == 0:
        print("No valid frames were processed from:", video_path)
        return

    average_prediction = np.mean(predictions, axis=0)

    predicted_class_index = np.argmax(average_prediction)
    predicted_class_name = CLASSES_LIST[predicted_class_index]
    confidence = average_prediction[predicted_class_index]

    print("Video:", video_path)
    print("Predicted Action:", predicted_class_name)
    print("Confidence:", confidence)
    print("-" * 40)

In [15]:
youtube_videos = [
    "Youtube_Videos/basketball.mp4",
    "Youtube_Videos/biking.mp4",
    "Youtube_Videos/tennis_swing.mp4"
]

for video_path in youtube_videos:
    predict_youtube_video(video_path)

Video: Youtube_Videos/basketball.mp4
Predicted Action: biking
Confidence: 0.5187436
----------------------------------------
Video: Youtube_Videos/biking.mp4
Predicted Action: tennis_swing
Confidence: 0.41136464
----------------------------------------
Video: Youtube_Videos/tennis_swing.mp4
Predicted Action: tennis_swing
Confidence: 0.7019732
----------------------------------------
